In [22]:
import math
import random
import warnings
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageDraw

try:
  import pandas as pd
except Exception:
  pd = None

try:
  from IPython.display import display
except Exception:
  display = print

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

warnings.filterwarnings("ignore", category=UserWarning)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


device: cpu


In [23]:
def to_numpy(x):
  """torch.Tensor 또는 numpy 배열을 numpy 배열로 변환한다."""
  if isinstance(x, torch.Tensor):
    return x.detach().cpu().numpy()
  return np.asarray(x)


def matrix_table(matrix, row_labels=None, col_labels=None, digits=3):
  """행렬을 그림이 아닌 표 형태로 확인한다."""
  arr = np.round(to_numpy(matrix), digits)
  if pd is not None:
    return pd.DataFrame(arr, index=row_labels, columns=col_labels)
  return arr


def show_matrix_table(title, matrix, row_labels=None, col_labels=None, digits=3):
  print(f"\n[{title}]")
  display(matrix_table(matrix, row_labels=row_labels, col_labels=col_labels, digits=digits))


def print_attention_topk(attn, query_labels, key_labels=None, top_k=2):
  """각 query가 가장 크게 참고한 key를 텍스트로 출력한다."""
  arr = to_numpy(attn)
  if key_labels is None:
    key_labels = query_labels
  for i, query in enumerate(query_labels):
    idx = np.argsort(arr[i])[::-1][:top_k]
    pairs = [f"{key_labels[j]}({arr[i, j]:.3f})" for j in idx]
    print(f"{query:>12s} ‑> " + ", ".join(pairs))


def move_batch_to_device(batch, device):
  """BatchEncoding, dict, tensor를 모두 안전하게 지정 device로 옮긴다."""
  if isinstance(batch, torch.Tensor):
    return batch.to(device)
  if hasattr(batch, "to"):
    try:
      return batch.to(device)
    except Exception:
      pass
  if isinstance(batch, dict):
    return {
      key: value.to(device) if hasattr(value, "to") else value
      for key, value in batch.items()
    }
  return batch


def meshgrid_ij(y, x):
  """torch 버전에 따라 indexing 인자 지원 여부가 달라서 안전하게 처리한다."""
  try:
    return torch.meshgrid(y, x, indexing="ij")
  except TypeError:
    return torch.meshgrid(y, x)

In [24]:
tokens = ["I", "love", "deep", "learning"]
seq_len = len(tokens)
d_model = 8

X = torch.randn(seq_len, d_model)
print("tokens:", tokens)
print("X shape:", X.shape)
print(X)

tokens: ['I', 'love', 'deep', 'learning']
X shape: torch.Size([4, 8])
tensor([[ 1.9269,  1.4873,  0.9007, -2.1055,  0.6784, -1.2345, -0.0431, -1.6047],
        [-0.7521,  1.6487, -0.3925, -1.4036, -0.7279, -0.5594, -0.7688,  0.7624],
        [ 1.6423, -0.1596, -0.4974,  0.4396, -0.7581,  1.0783,  0.8008,  1.6806],
        [ 1.2791,  1.2964,  0.6105,  1.3347, -0.2316,  0.0418, -0.2516,  0.8599]])


In [25]:
d_k = 8
d_v = 8

W_Q = torch.randn(d_model, d_k) * 0.5
W_K = torch.randn(d_model, d_k) * 0.5
W_V = torch.randn(d_model, d_v) * 0.5

Q = X @ W_Q
K = X @ W_K
V = X @ W_V

print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)


Q shape: torch.Size([4, 8])
K shape: torch.Size([4, 8])
V shape: torch.Size([4, 8])


In [26]:
def scaled_dot_product_attention(Q, K, V, mask=None):
  """
  Q: (seq_len, d_k)
  K: (seq_len, d_k)
  V: (seq_len, d_v)
  mask: (seq_len, seq_len), True/1이면 볼 수 있고 False/0이면 가린다.
  """

  d_k = Q.size(-1)
  scores = Q @ K.T / math.sqrt(d_k)

  if mask is not None:
    mask = mask.to(dtype=torch.bool, device=scores.device)
    scores = scores.masked_fill(~mask, float("-inf"))

  attention_weights = torch.softmax(scores, dim=-1)
  output = attention_weights @ V
  return output, attention_weights, scores

output, attention_weights, scores = scaled_dot_product_attention(Q, K, V)
print("scores shape:", scores.shape)
print("attention_weights shape:", attention_weights.shape)
print("output shape:", output.shape)
print("row sums:", attention_weights.sum(dim=-1))

scores shape: torch.Size([4, 4])
attention_weights shape: torch.Size([4, 4])
output shape: torch.Size([4, 8])
row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000])


In [27]:
show_matrix_table(
  "Raw scores: QK^T / sqrt(d_k)",
  scores,
  row_labels=tokens,
  col_labels=tokens,
)

show_matrix_table(
  "Attention weights: softmax(scores)",
  attention_weights,
  row_labels=tokens,
  col_labels=tokens,
)

print("\n각 query token이 가장 크게 참고한 token")
print_attention_topk(attention_weights, query_labels=tokens, key_labels=tokens, top_k=2)


[Raw scores: QK^T / sqrt(d_k)]


,I,love,deep,learning
I,4.274,-0.127,0.624,1.430
love,0.167,1.289,-0.959,-1.050
deep,-1.847,-0.352,1.489,0.527
learning,0.638,0.324,4.385,3.149



[Attention weights: softmax(scores)]


,I,love,deep,learning
I,0.912,0.011,0.024,0.053
love,0.213,0.655,0.069,0.063
deep,0.023,0.101,0.634,0.242
learning,0.018,0.013,0.751,0.218



각 query token이 가장 크게 참고한 token
           I ‑> I(0.912), learning(0.053)
        love ‑> love(0.655), I(0.213)
        deep ‑> deep(0.634), learning(0.242)
    learning ‑> deep(0.751), learning(0.218)


In [28]:
def attention_entropy(attn, eps=1e-9):
  return -(attn * (attn + eps).log()).sum(dim=-1).mean().item()


dims = [4, 8, 16, 32, 64, 128, 256]
rows = []

for d in dims:
  q = torch.randn(128, d)
  k = torch.randn(128, d)
  raw_scores = q @ k.T

  attn_no_scale = torch.softmax(raw_scores, dim=-1)
  attn_scale = torch.softmax(raw_scores / math.sqrt(d), dim=-1)

  rows.append({
    "d_k": d,
    "entropy_without_scaling": round(attention_entropy(attn_no_scale), 3),
    "entropy_with_scaling": round(attention_entropy(attn_scale), 3),
  })

if pd is not None:
  display(pd.DataFrame(rows))
else:
  for row in rows:
    print(row)

,d_k,entropy_without_scaling,entropy_with_scaling
0,4,3.371,4.370
1,8,2.487,4.377
2,16,1.622,4.367
3,32,1.027,4.356
4,64,0.636,4.361
5,128,0.388,4.334
6,256,0.294,4.362


In [29]:
encoder_output, encoder_attn, encoder_scores = scaled_dot_product_attention(Q, K, V)

show_matrix_table(
  "Encoder‑style attention: every token can attend to every token",
  encoder_attn,
  row_labels=tokens,
  col_labels=tokens,
)

print("\n행별 합:", encoder_attn.sum(dim=-1))



[Encoder‑style attention: every token can attend to every token]


,I,love,deep,learning
I,0.912,0.011,0.024,0.053
love,0.213,0.655,0.069,0.063
deep,0.023,0.101,0.634,0.242
learning,0.018,0.013,0.751,0.218



행별 합: tensor([1.0000, 1.0000, 1.0000, 1.0000])


In [30]:
causal_mask = torch.tril(torch.ones(seq_len, seq_len)).bool()
print("causal_mask: 1 = visible, 0 = hidden")
show_matrix_table("Causal mask", causal_mask.int(), row_labels=tokens, col_labels=tokens, digits=0)

masked_output, masked_attn, masked_scores = scaled_dot_product_attention(Q, K, V, mask=causal_mask)
show_matrix_table(
  "Decoder‑style attention after causal mask",
  masked_attn,
  row_labels=tokens,
  col_labels=tokens,
)
print("\n각 query token이 causal mask 이후 가장 크게 참고한 token")
print_attention_topk(masked_attn, query_labels=tokens, key_labels=tokens, top_k=2)


causal_mask: 1 = visible, 0 = hidden

[Causal mask]


,I,love,deep,learning
I,1,0,0,0
love,1,1,0,0
deep,1,1,1,0
learning,1,1,1,1



[Decoder‑style attention after causal mask]


,I,love,deep,learning
I,1.000,0.000,0.000,0.000
love,0.245,0.755,0.000,0.000
deep,0.030,0.133,0.837,0.000
learning,0.018,0.013,0.751,0.218



각 query token이 causal mask 이후 가장 크게 참고한 token
           I ‑> I(1.000), learning(0.000)
        love ‑> love(0.755), I(0.245)
        deep ‑> deep(0.837), love(0.133)
    learning ‑> deep(0.751), learning(0.218)


In [31]:
class MultiHeadSelfAttention(nn.Module):
  def __init__(self, d_model, num_heads):
    super().__init__()
    assert d_model % num_heads == 0
    self.d_model = d_model
    self.num_heads = num_heads
    self.head_dim = d_model // num_heads
    self.q_proj = nn.Linear(d_model, d_model)
    self.k_proj = nn.Linear(d_model, d_model)
    self.v_proj = nn.Linear(d_model, d_model)
    self.out_proj = nn.Linear(d_model, d_model)

  def forward(self, x, mask=None, return_attn=True):
    # x: (batch, seq_len, d_model)
    B, T, C = x.shape
    q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
    k = self.k_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
    v = self.v_proj(x).view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
    # q, k, v: (batch, heads, seq_len, head_dim)
    scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)
    # scores: (batch, heads, seq_len, seq_len)

    if mask is not None:
      mask = mask.to(dtype=torch.bool, device=x.device)
      scores = scores.masked_fill(~mask, float("‑inf"))

    attn = torch.softmax(scores, dim=-1)
    out = attn @ v
    # out: (batch, heads, seq_len, head_dim)

    out = out.transpose(1, 2).contiguous().view(B, T, C)
    out = self.out_proj(out)

    if return_attn:
      return out, attn
    return out
mha = MultiHeadSelfAttention(d_model=8, num_heads=4)
x_batch = X.unsqueeze(0)
mha_out, head_attn = mha(x_batch)

print("input:", x_batch.shape)
print("attention per head:", head_attn.shape)
print("output:", mha_out.shape)

input: torch.Size([1, 4, 8])
attention per head: torch.Size([1, 4, 4, 4])
output: torch.Size([1, 4, 8])


In [32]:
for h in range(head_attn.shape[1]):
  print(f"\n[Head {h}] 각 query token의 top‑2 attention")
  print_attention_topk(head_attn[0, h], query_labels=tokens, key_labels=tokens, top_k=2)



[Head 0] 각 query token의 top‑2 attention
           I ‑> deep(0.407), love(0.260)
        love ‑> love(0.278), I(0.270)
        deep ‑> learning(0.339), deep(0.275)
    learning ‑> learning(0.383), deep(0.366)

[Head 1] 각 query token의 top‑2 attention
           I ‑> deep(0.456), learning(0.264)
        love ‑> deep(0.337), learning(0.276)
        deep ‑> I(0.344), learning(0.251)
    learning ‑> I(0.263), learning(0.262)

[Head 2] 각 query token의 top‑2 attention
           I ‑> deep(0.553), love(0.255)
        love ‑> deep(0.382), love(0.311)
        deep ‑> deep(0.332), love(0.276)
    learning ‑> love(0.302), deep(0.282)

[Head 3] 각 query token의 top‑2 attention
           I ‑> I(0.305), love(0.288)
        love ‑> love(0.301), I(0.276)
        deep ‑> deep(0.324), learning(0.268)
    learning ‑> love(0.337), I(0.253)


In [33]:
def sinusoidal_positional_encoding(max_len, d_model):
  position = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
  even_dims = torch.arange(0, d_model, 2, dtype=torch.float32)
  div_term = torch.exp(even_dims * (-math.log(10000.0) / d_model))

  pe = torch.zeros(max_len, d_model)
  pe[:, 0::2] = torch.sin(position * div_term)
  pe[:, 1::2] = torch.cos(position * div_term)
  return pe
pe = sinusoidal_positional_encoding(max_len=64, d_model=32)
print(pe.shape)

torch.Size([64, 32])


In [34]:
show_matrix_table(
  "Positional encoding 첫 8개 위치와 첫 8개 차원",
  pe[:8, :8],
  row_labels=[f"pos {i}" for i in range(8)],
  col_labels=[f"dim {i}" for i in range(8)],
)

pe_norm = F.normalize(pe, dim=-1)
pe_similarity = pe_norm @ pe_norm.T

show_matrix_table(
  "Position 간 cosine similarity 일부",
  pe_similarity[:8, :8],
  row_labels=[f"pos {i}" for i in range(8)],
  col_labels=[f"pos {i}" for i in range(8)],
)


[Positional encoding 첫 8개 위치와 첫 8개 차원]


,dim 0,dim 1,dim 2,dim 3,dim 4,dim 5,dim 6,dim 7
pos 0,0.000,1.000,0.000,1.000,0.000,1.000,0.000,1.000
pos 1,0.841,0.540,0.533,0.846,0.311,0.950,0.177,0.984
pos 2,0.909,-0.416,0.902,0.431,0.591,0.807,0.348,0.937
pos 3,0.141,-0.990,0.993,-0.116,0.813,0.583,0.509,0.861
pos 4,-0.757,-0.654,0.778,-0.628,0.954,0.301,0.653,0.758
pos 5,-0.959,0.284,0.324,-0.946,1.000,-0.010,0.777,0.630
pos 6,-0.279,0.960,-0.230,-0.973,0.947,-0.321,0.876,0.483
pos 7,0.657,0.754,-0.714,-0.700,0.800,-0.599,0.947,0.320



[Position 간 cosine similarity 일부]


,pos 0,pos 1,pos 2,pos 3,pos 4,pos 5,pos 6,pos 7
pos 0,1.000,0.957,0.858,0.767,0.729,0.736,0.743,0.714
pos 1,0.957,1.000,0.957,0.858,0.767,0.729,0.736,0.743
pos 2,0.858,0.957,1.000,0.957,0.858,0.767,0.729,0.736
pos 3,0.767,0.858,0.957,1.000,0.957,0.858,0.767,0.729
pos 4,0.729,0.767,0.858,0.957,1.000,0.957,0.858,0.767
pos 5,0.736,0.729,0.767,0.858,0.957,1.000,0.957,0.858
pos 6,0.743,0.736,0.729,0.767,0.858,0.957,1.000,0.957
pos 7,0.714,0.743,0.736,0.729,0.767,0.858,0.957,1.000


In [35]:
try:
  from transformers import AutoTokenizer, AutoModel
  transformers_available = True

except Exception as e:
  transformers_available = False
  AutoTokenizer = None
  AutoModel = None
  print("transformers가 설치되어 있지 않습니다.")
  print("실제 BERT 예제는 첫 설치 셀 실행 후 다시 시도하세요.")
  print(type(e).__name__, e)


def load_auto_model_with_attention(model_name):
  """attention weight 반환을 위해 eager 구현을 우선 사용한다."""
  try:
    model = AutoModel.from_pretrained(
      model_name,
      output_attentions=True,
      attn_implementation="eager",
    )
  except TypeError:
    model = AutoModel.from_pretrained(
      model_name,
      output_attentions=True,
    )
  model.config.output_attentions = True
  return model.to(device).eval()

model_name_bert = "bert‑base‑multilingual‑cased"

if transformers_available:
  try:
    tokenizer_bert = AutoTokenizer.from_pretrained(model_name_bert)
    model_bert = load_auto_model_with_attention(model_name_bert)
    bert_available = True
    print("BERT model loaded")
  except Exception as e:
    bert_available = False
    tokenizer_bert = None
    model_bert = None
    print("BERT model load failed.")
    print("인터넷 연결 또는 모델 다운로드 상태를 확인하세요.")
    print(type(e).__name__, e)
else:
  bert_available = False
  tokenizer_bert = None
  model_bert = None

BERT model load failed.
인터넷 연결 또는 모델 다운로드 상태를 확인하세요.
OSError Repo id must use alphanumeric chars, '-', '_' or '.'. The name cannot start or end with '-' or '.' and the maximum length is 96: 'bert‑base‑multilingual‑cased'.


In [36]:
sentence = "강아지가 공을 물고 뛰었다. 그것은 매우 빨랐다."

if bert_available:
  encoded = tokenizer_bert(sentence, return_tensors="pt")
  encoded = move_batch_to_device(encoded, device)
  token_ids = encoded["input_ids"][0].detach().cpu().tolist()
  token_labels = tokenizer_bert.convert_ids_to_tokens(token_ids)
  print(token_labels)
else:
  # 모델 다운로드가 안 되는 환경에서 수업 흐름을 확인하기 위한 대체 token
  token_labels = ["[CLS]", "강아지", "공", "물고", "뛰었다", ".", "그것", "빠르다", "[SEP]"]
  print(token_labels)

['[CLS]', '강아지', '공', '물고', '뛰었다', '.', '그것', '빠르다', '[SEP]']


In [37]:
def make_fallback_bert_attention(token_labels):
  """모델 다운로드가 안 되거나 attention 반환이 실패할 때 사용하는 대체 attention."""
  n = len(token_labels)
  base = torch.eye(n) * 1.5 + torch.rand(n, n) * 0.3
  if n > 6:
    base[6, 1] += 1.2 # '그것'이 앞쪽 명사를 참고하는 예시 패턴
  return torch.softmax(base, dim=-1)

if bert_available:
  try:
    with torch.no_grad():
      bert_outputs = model_bert(**encoded)
    attentions = getattr(bert_outputs, "attentions", None)

    if attentions is None or len(attentions) == 0:
      raise RuntimeError("attention tensor가 반환되지 않았습니다.")

    print("number of layers:", len(attentions))
    print("last layer attention shape:", attentions[-1].shape)

    layer_idx = -1
    head_idx = 0
    bert_attn = attentions[layer_idx][0, head_idx].detach().cpu()

  except Exception as e:
    print("BERT attention 추출에 실패하여 대체 attention을 사용합니다.")
    print(type(e).__name__, e)
    attentions = None
    bert_attn = make_fallback_bert_attention(token_labels)
else:
  attentions = None
  bert_attn = make_fallback_bert_attention(token_labels)

show_matrix_table(
  "BERT attention example",
  bert_attn,
  row_labels=token_labels,
  col_labels=token_labels,
)
print("\n각 token의 top‑2 attention")
print_attention_topk(bert_attn, query_labels=token_labels, key_labels=token_labels, top_k=2)



[BERT attention example]


,[CLS],강아지,공,물고,뛰었다,.,그것,빠르다,[SEP]
[CLS],0.341,0.086,0.073,0.077,0.090,0.093,0.076,0.079,0.086
강아지,0.076,0.382,0.077,0.076,0.076,0.083,0.075,0.083,0.073
공,0.092,0.090,0.330,0.094,0.081,0.081,0.083,0.076,0.073
물고,0.069,0.090,0.075,0.375,0.078,0.069,0.084,0.080,0.079
뛰었다,0.090,0.079,0.076,0.080,0.338,0.076,0.078,0.092,0.092
.,0.074,0.075,0.087,0.072,0.069,0.386,0.081,0.072,0.085
그것,0.060,0.227,0.061,0.072,0.073,0.064,0.299,0.069,0.074
빠르다,0.081,0.081,0.084,0.087,0.079,0.080,0.068,0.364,0.076
[SEP],0.091,0.072,0.075,0.079,0.089,0.091,0.086,0.082,0.335



각 token의 top‑2 attention
       [CLS] ‑> [CLS](0.341), .(0.093)
         강아지 ‑> 강아지(0.382), 빠르다(0.083)
           공 ‑> 공(0.330), 물고(0.094)
          물고 ‑> 물고(0.375), 강아지(0.090)
         뛰었다 ‑> 뛰었다(0.338), [SEP](0.092)
           . ‑> .(0.386), 공(0.087)
          그것 ‑> 그것(0.299), 강아지(0.227)
         빠르다 ‑> 빠르다(0.364), 물고(0.087)
       [SEP] ‑> [SEP](0.335), .(0.091)


In [39]:
if bert_available and attentions is not None:
  mean_attn_last_layer = attentions[-1][0].mean(dim=0).detach().cpu()

else:
  n = len(token_labels)
  mean_attn_last_layer = bert_attn * 0.7 + torch.eye(n) / n
  mean_attn_last_layer = mean_attn_last_layer / mean_attn_last_layer.sum(
    dim=-1,
    keepdim=True,
  )


show_matrix_table(
  "BERT last‑layer mean attention over heads",
  mean_attn_last_layer,
  row_labels=token_labels,
  col_labels=token_labels,
)

print("\nhead 평균 기준 top‑2 attention")
print_attention_topk(
  mean_attn_last_layer,
  query_labels=token_labels,
  key_labels=token_labels,
  top_k=2,
)



[BERT last‑layer mean attention over heads]


,[CLS],강아지,공,물고,뛰었다,.,그것,빠르다,[SEP]
[CLS],0.431,0.074,0.063,0.067,0.078,0.080,0.066,0.068,0.075
강아지,0.065,0.467,0.066,0.066,0.065,0.071,0.064,0.072,0.063
공,0.080,0.078,0.422,0.081,0.070,0.070,0.071,0.066,0.063
물고,0.059,0.078,0.065,0.461,0.068,0.060,0.072,0.069,0.068
뛰었다,0.078,0.068,0.066,0.069,0.428,0.066,0.067,0.079,0.079
.,0.064,0.064,0.075,0.062,0.059,0.470,0.070,0.062,0.073
그것,0.052,0.196,0.053,0.062,0.063,0.055,0.395,0.060,0.064
빠르다,0.070,0.070,0.073,0.075,0.068,0.069,0.059,0.451,0.066
[SEP],0.078,0.062,0.065,0.068,0.077,0.078,0.074,0.071,0.426



head 평균 기준 top‑2 attention
       [CLS] ‑> [CLS](0.431), .(0.080)
         강아지 ‑> 강아지(0.467), 빠르다(0.072)
           공 ‑> 공(0.422), 물고(0.081)
          물고 ‑> 물고(0.461), 강아지(0.078)
         뛰었다 ‑> 뛰었다(0.428), [SEP](0.079)
           . ‑> .(0.470), 공(0.075)
          그것 ‑> 그것(0.395), 강아지(0.196)
         빠르다 ‑> 빠르다(0.451), 물고(0.075)
       [SEP] ‑> [SEP](0.426), .(0.078)


In [40]:
query_token_index = 0 # [CLS]
weights = mean_attn_last_layer[query_token_index].detach().cpu()
values, indices = torch.topk(weights, k=min(5, len(token_labels)))

print(f"Query token: {token_labels[query_token_index]}")
for rank, (value, idx) in enumerate(zip(values, indices), start=1):
  print(f"{rank}. {token_labels[idx]}: {value.item():.4f}")

Query token: [CLS]
1. [CLS]: 0.4309
2. .: 0.0799
3. 뛰었다: 0.0777
4. [SEP]: 0.0746
5. 강아지: 0.0738


In [41]:
from transformers import ViTImageProcessor, ViTModel

try:
    vit_import_available = True
except Exception as e:
    ViTImageProcessor = None
    ViTModel = None
    vit_import_available = False
    print("transformers가 설치되어 있지 않습니다.")
    print("실제 ViT 예제는 첫 설치 셀 실행 후 다시 시도하세요.")
    print(type(e).__name__, e)


def load_vit_with_attention(model_name):
    try:
        model = ViTModel.from_pretrained(
            model_name,
            output_attentions=True,
            attn_implementation="eager",
        )
    except TypeError:
        model = ViTModel.from_pretrained(
            model_name,
            output_attentions=True,
        )
    model.config.output_attentions = True
    return model.to(device).eval()


model_name_vit = "google/vit-base-patch16-224"

if vit_import_available:
    try:
        processor_vit = ViTImageProcessor.from_pretrained(model_name_vit)
        model_vit = load_vit_with_attention(model_name_vit)
        vit_available = True
        print("ViT model loaded")
    except Exception as e:
        processor_vit = None
        model_vit = None
        vit_available = False
        print("ViT model load failed.")
        print("인터넷 연결 또는 모델 다운로드 상태를 확인하세요.")
        print(type(e).__name__, e)
else:
    processor_vit = None
    model_vit = None
    vit_available = False

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ViT model loaded


In [42]:
from PIL import Image, ImageDraw

# 224 x 224 RGB 수업용 테스트 이미지
# 모델 입력용으로만 만들고, 별도로 출력하지 않는다.
image = Image.new("RGB", (224, 224), "white")
draw = ImageDraw.Draw(image)

draw.rectangle(
    [45, 65, 95, 160],
    fill=(120, 170, 255),
    outline=(30, 80, 180),
    width=3,
)

draw.ellipse(
    [125, 55, 185, 115],
    fill=(255, 190, 90),
    outline=(180, 90, 30),
    width=3,
)

draw.line([30, 190, 195, 175], fill=(60, 120, 60), width=6)

print("image size:", image.size)
print("image mode:", image.mode)

image size: (224, 224)
image mode: RGB


In [46]:
import math
import torch


def make_fallback_patch_attention(grid_size=14):
    """모델 다운로드가 안 되는 환경에서 사용하는 대체 patch attention."""
    yy, xx = meshgrid_ij(
        torch.arange(grid_size),
        torch.arange(grid_size),
    )

    patch_attention = torch.exp(-((xx - 8) ** 2 + (yy - 5) ** 2) / 12.0)
    return patch_attention / patch_attention.sum()


if vit_available:
    try:
        inputs_vit = processor_vit(images=image, return_tensors="pt")
        inputs_vit = move_batch_to_device(inputs_vit, device)

        with torch.no_grad():
            vit_outputs = model_vit(**inputs_vit)

        vit_attentions = getattr(vit_outputs, "attentions", None)

        if vit_attentions is None or len(vit_attentions) == 0:
            raise RuntimeError("attention tensor가 반환되지 않았습니다.")

        print("number of layers:", len(vit_attentions))
        # 기호 수정: ‑1 을 -1 로 변경
        print("last layer attention shape:", vit_attentions[-1].shape)

        last_attn = vit_attentions[-1][0]
        cls_to_patches = last_attn[:, 0, 1:].mean(dim=0).detach().cpu()
        num_patches = cls_to_patches.numel()
        grid_size = int(math.sqrt(num_patches))
        patch_attention = cls_to_patches.reshape(grid_size, grid_size)

    except Exception as e:
        print("ViT attention 추출에 실패하여 대체 patch attention을 사용합니다.")
        print(type(e).__name__, e)
        grid_size = 14
        patch_attention = make_fallback_patch_attention(grid_size)
else:
    grid_size = 14
    patch_attention = make_fallback_patch_attention(grid_size)

print("patch attention shape:", patch_attention.shape)

show_matrix_table(
    "ViT [CLS] attention to image patches 일부",
    patch_attention[:7, :7],
    row_labels=[f"row {i}" for i in range(7)],
    col_labels=[f"col {i}" for i in range(7)],
)

number of layers: 12
last layer attention shape: torch.Size([1, 12, 197, 197])
patch attention shape: torch.Size([14, 14])

[ViT [CLS] attention to image patches 일부]


,col 0,col 1,col 2,col 3,col 4,col 5,col 6
row 0,0.001,0.000,0.000,0.000,0.000,0.000,0.000
row 1,0.000,0.019,0.010,0.000,0.021,0.026,0.027
row 2,0.001,0.000,0.001,0.002,0.004,0.005,0.002
row 3,0.000,0.002,0.002,0.004,0.003,0.002,0.001
row 4,0.000,0.001,0.002,0.007,0.004,0.006,0.002
row 5,0.007,0.001,0.003,0.007,0.006,0.007,0.002
row 6,0.002,0.001,0.003,0.008,0.006,0.009,0.004


In [47]:
flat = patch_attention.flatten()
values, indices = torch.topk(flat, k=5)

print("[CLS] token이 가장 크게 참고한 patch 좌표 top‑5")
for rank, (value, idx) in enumerate(zip(values, indices), start=1):
  row = int(idx.item() // grid_size)
  col = int(idx.item() % grid_size)
  print(f"{rank}. patch(row={row}, col={col}) attention={value.item():.5f}")

[CLS] token이 가장 크게 참고한 patch 좌표 top‑5
1. patch(row=1, col=6) attention=0.02741
2. patch(row=1, col=5) attention=0.02556
3. patch(row=13, col=6) attention=0.02429
4. patch(row=1, col=7) attention=0.02417
5. patch(row=1, col=11) attention=0.02399


In [48]:
import torch
import torch.nn as nn


class MiniTransformerEncoderBlock(nn.Module):

    def __init__(self, d_model, num_heads, d_ff=32, dropout=0.1):
        super().__init__()
        # 주의: MultiHeadSelfAttention 클래스가 사전에 정의되어 있어야 합니다.
        self.attn = MultiHeadSelfAttention(d_model=d_model, num_heads=num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_out, attn_weights = self.attn(x, mask=mask, return_attn=True)
        x = self.norm1(x + self.dropout(attn_out))
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_out))
        return x, attn_weights

block = MiniTransformerEncoderBlock(d_model=8, num_heads=4)
block_out, block_attn = block(x_batch)

print("input:", x_batch.shape)
print("output:", block_out.shape)
print("attention:", block_attn.shape)

input: torch.Size([1, 4, 8])
output: torch.Size([1, 4, 8])
attention: torch.Size([1, 4, 4, 4])


In [49]:
encoder_layer = nn.TransformerEncoderLayer(
  d_model=8,
  nhead=4,
  dim_feedforward=32,
  batch_first=True,
)


pt_out = encoder_layer(x_batch)
print("PyTorch TransformerEncoderLayer output:", pt_out.shape)


PyTorch TransformerEncoderLayer output: torch.Size([1, 4, 8])
